# Correct the input boundaries of the Unincorporated counties


In [2]:
base_url = 'https://api.tdei.us'
datasets_path = '/api/v1/datasets'
auth_path = '/api/v1/authenticate'
username = 'nareshd@gaussiansolutions.com'
password = 'a$hwa7hamA'

In [3]:
import requests

url = base_url + auth_path
payload = {
    'username': username,
    'password': password
}
headers = {
    'Content-Type': 'application/json'
}

response = requests.post(url, json=payload, headers=headers)
response.json()
access_token = response.json()['access_token']

In [ ]:
# Get the current boundary from the TDEI system and save it to .json file
import requests
import json
import os
import shutil
# api_key = '99b9b841-0ae9-45fc-9ca2-e6209b2a512b'
# dataset_id = '5aac6caf-abd6-4453-8b8d-058c0de553a3'
metadata_folder = '../ui-union/metadata'
existing_boundaries_folder = '../ui-union/existing_boundaries'
work_folder = '../ui-union/work'
url = (f'https://api.tdei.us/api/v1/datasets?'
'page_no=1&page_size=10&sort_field=uploaded_timestamp&sort_order=DESC&status=All&'
'tdei_dataset_id=1e7ed732-f470-4ae9-bfca-2e9c1e3fc6ac'
'&tdei_service_id=a008c57d-7959-478d-97e3-b3ca4268eaa6&tdei_project_group_id=1dd7c38e-c7a6-4e3a-be8b-379f823a7ad7')
response = requests.get(url, headers={'Authorization': 'Bearer ' + access_token})
data = response.json()
if len(data) > 0:
    metadata = data[0]['metadata']
    # dump it in the file
    name = metadata['dataset_detail']['name']
    name = name.replace('GS','').replace('UI','').replace('_','')
    name = name.replace(' ', '_')
    name = name.lower()
    name = name.replace('county','')
    file_path = metadata_folder + '/' + name + '.json'
    print(name)
    with open(file_path, 'w') as f:
        json.dump(metadata, f)
    # boundary to be got
    boundary = metadata['dataset_detail']['dataset_area']
    boundary_file_name = name+'_before.geojson'
    boundary_path = os.path.join(existing_boundaries_folder,boundary_file_name)
    with open(boundary_path,'w') as f:
        json.dump(boundary,f)
    work_file_name = name+'_boundary.geojson'
    work_file_path = os.path.join(work_folder,work_file_name)
    shutil.copy(boundary_path,work_file_path)
else:
    print('No data found')

whatcom


In [199]:
import json
input_file = work_file_path
with open(input_file, 'r') as f:
    data = json.load(f)

# print(data['features'][0]['geometry']['coordinates'])
coordinates = data['features'][0]['geometry']['coordinates']
for coord in coordinates:
    if len(coord) > 1:
        for item in coord:
            if len(item) == 1:
                print('www')
                print(item)
    else:
        if len(coord[0]) == 1:
            print(len(coord[0]))
            print('www')

In [200]:
import geopandas as gpd
import json
input_file = work_file_path
input_gdf = gpd.read_file(input_file)
input_gdf.columns

# first geometry
county_shape_current = input_gdf.iloc[0].geometry

In [201]:
from shapely.validation import explain_validity
print(county_shape_current.is_valid)
# Example: "Self-intersection[10.5 20.2]"
print(explain_validity(county_shape_current))

True
Valid Geometry


In [202]:
from shapely.validation import explain_validity
exploded_gdf = input_gdf.explode()
def is_valid(shape):
    try:
        shape.is_valid
        if not shape.is_valid:
            print(explain_validity(shape))
        return shape.is_valid
    except Exception:
        return False
exploded_gdf['is_valid'] = exploded_gdf['geometry'].apply(is_valid)
# check count of invalid polygons
invalid_polygons_gdf = exploded_gdf[exploded_gdf['is_valid'] == False]
print(len(invalid_polygons_gdf))
# invalid_polygons_gdf.to_file('../ui-union/working/king_ui_boundary_invalid.geojson')  

0
